# VolleyVision: retrain YOLO (fine-tune `best4.pt` on old + checked data)

Continues from your current ball model (`best4.pt`: YOLO26n, 1280 px) on:
- **`volleyball_combined`**: your Roboflow data, the same set `best4.pt` was trained on
- **`yolo_vN`**: frames from your checked clips (`python -m tools.export_training --name vN` on the Mac)

**Fair scoring:** frames from your 10 test clips (`data/gt`: test7, 10, 12, 17, 21, 29, 30, 34, 36, 42) are left out of
training here. Three of them (test7, 12, 17) are in `volleyball_combined`, and `best4.pt` did train on them.

**Put in Google Drive under `MyDrive/VolleyVision/`:**
- `volleyball_combined.zip`: zip of `training/datasets/volleyball_combined`
- `yolo_vN.zip`: zip of `training/datasets/yolo_vN` (e.g. `yolo_v2.zip`)
- `best4.pt`: from `models/`

Runtime → Change runtime type → **GPU** (T4 is fine; L4/A100 faster), then Run all.
Weights are saved to `MyDrive/VolleyVision/yolo_ft/` after every epoch.
Afterwards, score the new weights against the current ones with `python -m tools.scoreboard --yolo <new.pt>` on the Mac.


In [ ]:
# ---- settings ----
EXPORT = 'yolo_v2'          # the export to add (name of the zip without .zip)
EPOCHS = 30                 # fine-tuning: fewer epochs than from scratch
BATCH = 16                  # T4 at 1280 px; use 32 on L4/A100
EXCLUDE_TEST_CLIPS = True   # keep your 10 test clips out of training (fair scoring)
TEST_CLIPS = ['test7', 'test10', 'test12', 'test17', 'test21', 'test29', 'test30', 'test34', 'test36', 'test42']

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/VolleyVision'
import os
for f in ['volleyball_combined.zip', f'{EXPORT}.zip', 'best4.pt']:
    print(('OK      ' if os.path.exists(f'{D}/{f}') else 'MISSING ') + f)

In [ ]:
!pip install -q ultralytics
# unzip both datasets side by side (data_with_combined.yaml refers to ../volleyball_combined)
!mkdir -p /content/datasets && cd /content/datasets && unzip -q -o {D}/volleyball_combined.zip && unzip -q -o {D}/{EXPORT}.zip
!ls /content/datasets

In [ ]:
# Find both dataset folders (zips may or may not contain the top folder) and build the training lists
import glob, re, yaml
def find(name):
    for p in [f'/content/datasets/{name}', f'/content/datasets/{name}/{name}', '/content/datasets']:
        if os.path.isdir(f'{p}/train/images'):
            return p
    hits = glob.glob(f'/content/datasets/**/{name}', recursive=True)
    return hits[0] if hits else None
OLD, NEW = find('volleyball_combined'), find(EXPORT)
print('old:', OLD, '| new:', NEW)

def clip_of(path):                      # Roboflow: test7_123_jpg.rf.xxx.jpg ; export: test47_123.jpg
    m = re.match(r'(.+?)_(\d+)(?:_jpg\.rf\..*|\.jpg|\.png)$', os.path.basename(path))
    return m.group(1) if m else None

lists = {}
for split, sub in (('train', 'train'), ('val', 'valid')):
    imgs = []
    for root in (OLD, NEW):
        imgs += sorted(glob.glob(f'{root}/{sub}/images/*'))
    if EXCLUDE_TEST_CLIPS:
        before = len(imgs)
        imgs = [p for p in imgs if clip_of(p) not in TEST_CLIPS]
        print(f'{split}: left out {before - len(imgs)} images from test clips')
    lists[split] = f'/content/datasets/{split}_list.txt'
    open(lists[split], 'w').write('\n'.join(imgs) + '\n')
    print(f'{split}: {len(imgs)} images')
yaml.safe_dump({'train': lists['train'], 'val': lists['val'], 'nc': 1, 'names': ['volleyball']},
               open('/content/datasets/data.yaml', 'w'))

## Fine-tune
Same recipe as `best4.pt` (MuSGD, 1280 px, no mosaic, mixup 0.1), lower starting learning rate since it continues
from trained weights. Best weights: `MyDrive/VolleyVision/yolo_ft/<run>/weights/best.pt`.

In [ ]:
from ultralytics import YOLO
model = YOLO(f'{D}/best4.pt')
model.train(data='/content/datasets/data.yaml', epochs=EPOCHS, imgsz=1280, batch=BATCH, optimizer='MuSGD',
            lr0=0.002, cos_lr=True, patience=10, mosaic=0.0, mixup=0.1, cls=0.5, box=7.5,
            project=f'{D}/yolo_ft', name=EXPORT, exist_ok=True, save_period=1, workers=2)

## Quick check
Ultralytics' own validation (box mAP on the validation images). The number that matters is the pipeline score on your
test clips: copy `best.pt` to `models/` on the Mac (e.g. `models/best5.pt`) and run `python -m tools.scoreboard --yolo models/best5.pt`.

In [ ]:
best = f'{D}/yolo_ft/{EXPORT}/weights/best.pt'
m = YOLO(best).val(data='/content/datasets/data.yaml', imgsz=1280, batch=BATCH)
print(f'mAP50 {m.box.map50:.3f}   precision {m.box.mp:.3f}   recall {m.box.mr:.3f}')
old = YOLO(f'{D}/best4.pt').val(data='/content/datasets/data.yaml', imgsz=1280, batch=BATCH)
print(f'best4 on the same validation set: mAP50 {old.box.map50:.3f}   precision {old.box.mp:.3f}   recall {old.box.mr:.3f}')

## If the runtime disconnected
Re-run the cells above **Fine-tune**, then this cell: it continues from the last saved epoch.

In [ ]:
from ultralytics import YOLO
YOLO(f'{D}/yolo_ft/{EXPORT}/weights/last.pt').train(resume=True)